# Module 15 — Spark Scala (silver → gold parquet)

Backfill **volume** : le poll Python reste le quotidien. Spark **projette** le silver JSON en parquet.

**Prérequis** : MinIO up, articles `parsed` (`uv run presslake parse`).

Tuto : [`docs/modules/15-spark-scala.md`](../docs/modules/15-spark-scala.md) · ADR [0004](../docs/adr/0004-un-seul-moteur-jvm.md) · [0009](../docs/adr/0009-gold-parquet-silver.md)

## Cas A — CLI et contrat gold (sans lancer le cluster)

In [1]:
import json
from presslake.contracts.validate import CONTRACTS_DIR, validate_gold
from presslake.spark_job.run import compose_run_argv, default_input_uri, default_output_uri

sample = json.loads((CONTRACTS_DIR / "examples" / "gold.sample.json").read_text(encoding="utf-8"))
validate_gold(sample)
print("gold sample OK", sample["feed_id"], sample["dt"])
print("input ", default_input_uri())
print("output", default_output_uri())
print("cmd   ", " ".join(compose_run_argv(input_uri=default_input_uri(), output_uri=default_output_uri(), build=False)))


gold sample OK france24 2026-08-31
input  s3a://presslake/silver
output s3a://presslake/gold/layer=silver_parquet
cmd    docker compose --profile spark run --rm -e SPARK_INPUT=s3a://presslake/silver -e SPARK_OUTPUT=s3a://presslake/gold/layer=silver_parquet spark-backfill


## Cas B — Le gold n'est pas le RAG

`search` MCP / `retrieve` lisent OpenSearch + Qdrant. Le parquet gold est une **copie colonnaire** pour le scan volume.

In [3]:
from presslake.spark_job.run import list_gold_keys

try:
    keys = list_gold_keys(max_keys=20)
except Exception as exc:
    print("MinIO inaccessible (normal si compose est down) :", type(exc).__name__, exc)
else:
    print(f"{len(keys)} objet(s) sous gold/")
    for k in keys[:15]:
        print(" ", k)
    if not keys:
        print("Lancer : uv run presslake spark --build")


20 objet(s) sous gold/
  gold/layer=silver_parquet/_SUCCESS
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-28/part-00006-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-29/part-00000-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-29/part-00006-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-30/part-00001-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-30/part-00009-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-31/part-00000-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-world/dt=2026-08-31/part-00001-99da99b7-97ff-475c-b8c6-27d1cfa02521.c000.snappy.parquet
  gold/layer=silver_parquet/feed_id=bbc-worl